# NB6 — 무릎각 시계열 + far 보간/라벨 **[CPU 가능]**
4/20 D 단계 실험 8·9 의 최종 체인을 그대로 실행 (회귀 테스트: 4/20 입력으로 기존 결과와 전 키 일치 확인됨).
1. `knee_measurement_basis.py` — 프레임별 near/far 무릎각 + 표면 측정 근거(residual, npts) → `knee_basis_full`
2. `far_thigh_npts_gate.py` — far thigh 관측량 분포의 골 → `N_low` (붕괴 프레임 판정)
3. `far_thigh_infill.py` — 붕괴 gap 3분기 보간 (선형 / cubic-quad4 / cubic-peak) → `far_knee_final`
4. `far_thigh_infill_v2.py` — σ(Δ) 캘리브레이션 + 최종 저장

산출 `sessions/{SESSION}/05_knee/far_knee_final.{npz,csv}`: `fid, knee_near, knee_far_final, label(MEASURED/INTERPOLATED/INTERPOLATED_HIGH_CURV), sigma_interp`

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_full_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
#@title 1. 보행주기(프레임) 측정 → WALK_GAIT_FRAMES
!pip -q install smplx
import numpy as np, json
n = np.load(walkcfg.NEAR_F3)
k = n["knee_fit"] - n["knee_fit"].mean()
ac = np.correlate(k, k, "full")[len(k)-1:]; ac /= ac[0]
_m = json.load(open(f"{walkcfg.EXTRACT_DIR}/meta.json"))
fps = (_m.get("window") or {}).get("fps_eff") or _m["fps"]   # STEP=2 면 30
lo, hi = int(0.7*fps), min(len(ac)-1, int(2.0*fps))
L = lo + int(np.argmax(ac[lo:hi]))
os.environ["WALK_GAIT_FRAMES"] = str(L); importlib.reload(walkcfg)
print(f"보행주기 = {L} 프레임 = {L/fps:.2f}s  (자기상관 {ac[L]:.2f}; 4/20 = 41.5f @30fps)")

In [ ]:
#@title 2. 측정 근거 (full)
!cd "{ROOT}/scripts" && python knee_measurement_basis.py 2>&1 | tail -25

In [ ]:
#@title 3. far 붕괴 게이트 → 4. 보간 v1 → 5. 보간 v2 (최종)
!cd "{ROOT}/scripts" && python far_thigh_npts_gate.py 2>&1 | tail -15
!cd "{ROOT}/scripts" && python far_thigh_infill.py 2>&1 | tail -15
!cd "{ROOT}/scripts" && python far_thigh_infill_v2.py 2>&1 | tail -25

In [ ]:
#@title 6. 결과
from IPython.display import Image, display
f = np.load(f"{walkcfg.KNEE_DIR}/far_knee_final.npz", allow_pickle=True)
u, c = np.unique(f["label"].astype(str), return_counts=True); print(dict(zip(u, c)))
print(f"near ROM {np.ptp(f['knee_near']):.1f}°  far ROM {np.ptp(f['knee_far_final']):.1f}°")
for p in ("far_knee_final_v2.png", "far_knee_final.png"):
    fp = f"{walkcfg.KNEE_DIR}/{p}"
    if os.path.exists(fp): display(Image(fp)); break

## 시각화 (NB6)

In [ ]:
#@title 시각화 — NB6 결과 확인 (viz/NB6/  + viz/videos/)
FIDS = "5660 5750"   #@param {type:"string"}
N_EVEN = 0           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 6 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 6 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB6/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB6_knee.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))